## Import the necessary packages

In [1]:
import pandas as pd
import numpy as np
from peshbeen.models import ms_arr
from peshbeen.transformations import rolling_quantile, fourier_terms

from peshbeen.metrics import MSE
from sklearn.preprocessing import OneHotEncoder
import pickle

import zipfile
import os

## Concatenation of train data and validation data, and data processing

In [2]:
def load_zip_dataset(file_path: str, target_filename: str) -> pd.DataFrame:
    """
    Safely loads a specific CSV from a ZIP archive to avoid 
    'Multiple files found' errors.
    """
    try:
        with zipfile.ZipFile(file_path, 'r') as z:
            # Open specifically the file we want
            with z.open(target_filename) as f:
                return pd.read_csv(f)
    except Exception as e:
        print(f"Error loading {target_filename}: {e}")
        raise
    
## Load and preprocess data
# Define the specific CSV name inside ZIP
target_csv_val = "turingAI_forecasting_challenge_validation_dataset.csv"
path_val = "../data/turingAI_forecasting_challenge_validation_dataset.csv.zip"
target_csv_train = "turingAI_forecasting_challenge_dataset.csv"
path_train = "../data/turingAI_forecasting_challenge_dataset.zip"
ead_df = load_zip_dataset(path_train, target_csv_train)
ead_valid = load_zip_dataset(path_val, target_csv_val)
ead_df = pd.concat([ead_df, ead_valid])

ead_df = ead_df[(ead_df["variable_type"] == "outcome")&(ead_df["value"] != -9999)][["dt", "value"]]
ead_df.rename(columns={"dt": "date", "value": "ead"}, inplace=True) # rename columns for clarity
ead_df["date"] = pd.to_datetime(ead_df["date"], format="mixed", utc=True).dt.tz_convert(None).dt.normalize()
## sort by date
ead_df.sort_values("date", inplace=True)

## add day of week and month as features
ead_df["day"] = ead_df["date"].dt.dayofweek
ead_df["month"] = ead_df["date"].dt.month
ead_df.set_index("date", inplace=True)


## Add public holidays and after-holiday indicators
england_public_holidays = {
    "2021-01-01": "New Year's Day",
    "2021-04-02": "Good Friday",
    "2021-04-05": "Easter Monday",
    "2021-05-03": "Early May bank holiday",
    "2021-05-31": "Spring bank holiday",
    "2021-08-30": "Summer bank holiday",
    "2021-12-27": "Christmas Day (substitute day)",
    "2021-12-28": "Boxing Day (substitute day)",

    "2022-01-03": "New Year's Day (substitute day)",
    "2022-04-15": "Good Friday",
    "2022-04-18": "Easter Monday",
    "2022-05-02": "Early May bank holiday",
    "2022-06-02": "Spring bank holiday",
    "2022-06-03": "Platinum Jubilee bank holiday",
    "2022-08-29": "Summer bank holiday",
    "2022-12-26": "Boxing Day",
    "2022-12-27": "Christmas Day (substitute day)",

    "2023-01-02": "New Year's Day (substitute day)",
    "2023-04-07": "Good Friday",
    "2023-04-10": "Easter Monday",
    "2023-05-01": "Early May bank holiday",
    "2023-05-29": "Spring bank holiday",
    "2023-08-28": "Summer bank holiday",
    "2023-12-25": "Christmas Day",
    "2023-12-26": "Boxing Day",

    "2024-01-01": "New Year's Day",
    "2024-03-29": "Good Friday",
    "2024-04-01": "Easter Monday",
    "2024-05-06": "Early May bank holiday",
    "2024-05-27": "Spring bank holiday",
    "2024-08-26": "Summer bank holiday",
    "2024-12-25": "Christmas Day",
    "2024-12-26": "Boxing Day",

    "2025-01-01": "New Year's Day",
    "2025-04-18": "Good Friday",
    "2025-04-21": "Easter Monday",
    "2025-05-05": "Early May bank holiday",
    "2025-05-26": "Spring bank holiday",
    "2025-08-25": "Summer bank holiday",
    "2025-12-25": "Christmas Day",
    "2025-12-26": "Boxing Day",

    "2026-01-01": "New Year's Day",
    "2026-04-03": "Good Friday",
    "2026-04-06": "Easter Monday",
    "2026-05-04": "Early May bank holiday",
    "2026-05-25": "Spring bank holiday",
    "2026-08-31": "Summer bank holiday",
    "2026-12-25": "Christmas Day",
    "2026-12-28": "Boxing Day (substitute day)",
}
pb_holidays = pd.DataFrame(list(england_public_holidays.items()), columns=['date', 'holiday_name'])
pb_holidays["date"] = pd.to_datetime(pb_holidays["date"])
pb_holidays["holiday"] = 1

# Create the after-holiday dataframe in one clean step to avoid SettingWithCopy warnings
after_holiday = pd.DataFrame({
    "date": pb_holidays["date"] + pd.Timedelta(days=1),
    "holiday": 2
})


## import outliers dates list from pickle
    
with open('notebook_param_res/outliers_dates.pkl', 'rb') as f:
    outliers_dates = pickle.load(f)

# Create a DataFrame for outliers which corresponds to operational pressure days
outliers_df = pd.DataFrame({
    "date": pd.to_datetime(outliers_dates),
    "holiday": 3
})


# Combine outliers_df with pb_holidays and after_holiday, ensuring that the 'holiday' column is correctly assigned
holiday_df = pd.concat([pb_holidays[["date", "holiday"]], after_holiday, outliers_df], ignore_index=True)

# Note the reassignment back to holiday_df here!
# This resolves consecutive holidays (e.g. Boxing day).
# Sorting by [Date, Holiday] ensures '1' comes before '2' for the same date.
# Dropping duplicates keeps the '1' and throws away the '2'.
holiday_df = holiday_df.sort_values(by=["date", "holiday"], ascending=[True, True]) \
                       .drop_duplicates(subset=['date'], keep='first') \
                       .reset_index(drop=True)
holiday_df.set_index("date", inplace=True)
target_var = "ead"

ead_df = ead_df.merge(holiday_df, left_index=True, right_index=True, how="left")

ead_df["holiday"].fillna(0, inplace=True)
cat_vars = ["day", "month", "holiday"]
ead_df[cat_vars] = ead_df[cat_vars].astype('int').astype('category')

## Add Fourier terms for yearly cycle
ead_dff = ead_df.merge(fourier_terms(ead_df.index, period=365.25, num_terms=1), left_index=True, right_index=True)
ead_dff.drop(columns = "cos_1_365.25", inplace=True)

## Run the modeling algorithm pipeline

In [3]:
n_splits = 131
H = 10

## One-Hot Encoder for categorical variables
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
## Lag transformation function - rolling quantile with window size 49, quantile 0.2, shifted by 10 to avoid data leakage, and minimum samples of 49 to ensure stability of the quantile estimate
transformation_function = [rolling_quantile(window_size=49, quantile=0.2, shift=10, min_samples=49)]

## Model specification and cross-validation
ms_mod = ms_arr(n_components=2, lags=[4, 8, 10], target_col=target_var, cat_variables=cat_vars,
    categorical_encoder=ohe_encoder, n_iter=100, random_state=42, lag_transform=transformation_function)

## Fit the model on the training data (excluding the last n_splits + H observations to avoid data leakage during cross-validation) for Markov-Switching model so the models learns parameters based on new data and not just the initial training period, which may not be representative of the entire dataset. This is especially important for time series data where patterns can change over time.
ms_mod.fit(ead_dff[:-n_splits-H])
cv_df = ms_mod.cross_validate(ead_dff,
                        cv_split=n_splits, step_size=1, test_size=10,
                        metrics=[MSE], h_split_point=5, n_iter=60)


## Generating the submission files and saving it to the submission folder

In [4]:
cv_df["MSE"] = (cv_df["y_true"] - cv_df["y_pred"]) ** 2
cv_df["horizon_cat"]=np.where(cv_df["horizon"]<=5, "mse_1_5", "mse_6_10")

forecasts_df = cv_df.pivot_table(index="cutoff", columns="horizon", values="y_pred")
forecasts_df.rename(columns=lambda x: f"day_{x}", inplace=True)
forecasts_df.reset_index(drop=True, inplace=True)
forecasts_df_final = forecasts_df.reset_index()
forecasts_df_final.rename(columns={"index": "forecast_id"}, inplace=True)
forecasts_df_final["forecast_id"] = forecasts_df_final["forecast_id"]+1
forecasts_df_final.to_csv("../submission/pred_matrix.csv", index=False)

mse_df = cv_df.pivot_table(index="cutoff", columns="horizon_cat", values="MSE", aggfunc="mean")
mse_df=mse_df.reset_index(drop=True)
mse_df.reset_index(drop=False, inplace=True) # reset index to get forecast_id as a column
mse_df.rename(columns={"index": "forecast_id"}, inplace=True)
mse_df["forecast_id"] = mse_df["forecast_id"]+1
mse_df.to_csv("../submission/mse_summary.csv", index=False)